# Can we predict 12-month loan default?

**Question:** using application-time information, can we predict `defaulted` (1 = defaulted within 12 months of funding)?

**Plan:** check data quality → explore → hold out a test set → compare a baseline against a few models with cross-validation → evaluate once on the test set → look at which features matter and whether the predictions are usable.

Because only ~10% of loans default, accuracy is misleading (always predicting "no default" gets ~90%). Primary metrics are **ROC AUC** and **PR AUC** (average precision), compared with the no-skill PR AUC = base rate.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
                             brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED=0
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); cols=pd.read_csv('columns.csv')
cols

## 1. Data quality

In [ ]:
print(df.shape)
print('missing values:', int(df.isna().sum().sum()), '| duplicate rows:', df.duplicated().sum(), '| duplicate ids:', df.application_id.duplicated().sum())
display(df.describe().T)
print(df.defaulted.value_counts(normalize=True).rename('share'))

`application_id` is an identifier, not a feature; I drop it (and check below that it carries no signal). There is no date column, so a time-based split isn't possible; a stratified random split is used, which assumes future applicants resemble these.

## 2. Exploration

In [ ]:
num=['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig,ax=plt.subplots(2,3,figsize=(14,7))
for a,c in zip(ax.ravel(),num):
    q=pd.qcut(df[c],10,duplicates='drop')
    r=df.groupby(q,observed=True).defaulted.mean()
    a.bar(range(len(r)),r.values,color='#4c78a8'); a.axhline(df.defaulted.mean(),color='k',ls='--',lw=1)
    a.set_title(f'default rate by decile of {c}'); a.set_xlabel('decile (low → high)'); a.set_ylabel('default rate')
plt.tight_layout(); plt.show()
print(df.groupby('region').defaulted.agg(['mean','size']))
print(df[num+['defaulted']].corr(method='spearman')['defaulted'].drop('defaulted').round(3))
print('Spearman corr of application_id with defaulted:', round(df[['application_id','defaulted']].corr(method='spearman').iloc[0,1],3))

## 3. Hold-out split and models

In [ ]:
X=df.drop(columns=['defaulted','application_id']); y=df.defaulted
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,stratify=y,random_state=SEED)
print(len(Xtr),len(Xte),'train/test default rate:',ytr.mean().round(4),yte.mean().round(4))
pre=ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(),['region'])])
models={
 'baseline (prior)':make_pipeline(pre,DummyClassifier(strategy='prior')),
 'logistic regression':make_pipeline(pre,LogisticRegression(max_iter=1000)),
 'random forest':make_pipeline(pre,RandomForestClassifier(300,min_samples_leaf=20,n_jobs=-1,random_state=SEED)),
 'gradient boosting':make_pipeline(pre,HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,random_state=SEED)),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
rows=[]
for n,m in models.items():
    s=cross_validate(m,Xtr,ytr,cv=cv,scoring={'roc_auc':'roc_auc','pr_auc':'average_precision'})
    rows.append({'model':n,'ROC AUC':s['test_roc_auc'].mean(),'ROC sd':s['test_roc_auc'].std(),'PR AUC':s['test_pr_auc'].mean(),'PR sd':s['test_pr_auc'].std()})
cvres=pd.DataFrame(rows).set_index('model').round(3); cvres

Cross-validation uses the training set only; the test set has not been touched yet.

## 4. Test-set evaluation

In [ ]:
res={}
for n,m in models.items():
    m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]
    res[n]=dict(p=p,roc=roc_auc_score(yte,p),pr=average_precision_score(yte,p),brier=brier_score_loss(yte,p))
print(pd.DataFrame({n:{k:v for k,v in r.items() if k!='p'} for n,r in res.items()}).T.round(3))
print('no-skill PR AUC (base rate):',round(yte.mean(),3))
rng=np.random.default_rng(SEED); p=res['logistic regression']['p']; yv=yte.values
bs=[roc_auc_score(yv[i],p[i]) for i in (rng.integers(0,len(yv),len(yv)) for _ in range(500))]
print('logistic ROC AUC 95%% CI: %.3f-%.3f'%tuple(np.percentile(bs,[2.5,97.5])))

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
for n,r in res.items():
    if n.startswith('baseline'): continue
    f,t,_=roc_curve(yte,r['p']); ax[0].plot(f,t,label=f"{n} ({r['roc']:.3f})")
    pr,rc,_=precision_recall_curve(yte,r['p']); ax[1].plot(rc,pr,label=f"{n} ({r['pr']:.3f})")
ax[0].plot([0,1],[0,1],'k--',lw=1); ax[0].set(title='ROC',xlabel='false positive rate',ylabel='true positive rate')
ax[1].axhline(yte.mean(),color='k',ls='--',lw=1); ax[1].set(title='Precision-recall',xlabel='recall',ylabel='precision')
for n in ['logistic regression','gradient boosting']:
    fp,mp=calibration_curve(yte,res[n]['p'],n_bins=8,strategy='quantile'); ax[2].plot(mp,fp,'o-',label=n)
ax[2].plot([0,.5],[0,.5],'k--',lw=1); ax[2].set(title='Calibration',xlabel='mean predicted',ylabel='observed rate')
for a in ax: a.legend()
plt.tight_layout(); plt.show()

## 5. What drives the predictions, and is it usable?

In [ ]:
best=models['logistic regression']
pi=permutation_importance(best,Xte,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
imp=pd.Series(pi.importances_mean,index=Xte.columns).sort_values()
imp.plot.barh(xerr=pd.Series(pi.importances_std,index=Xte.columns)[imp.index],color='#4c78a8')
plt.xlabel('drop in test ROC AUC when shuffled'); plt.title('Permutation importance (logistic regression)'); plt.show()
names=best[0].get_feature_names_out()
print(pd.Series(best[-1].coef_[0],index=names).sort_values().round(3))

In [ ]:
p=res['logistic regression']['p']
d=pd.DataFrame({'p':p,'y':yte.values}); d['bucket']=pd.qcut(d.p,5,labels=['1 (lowest risk)','2','3','4','5 (highest risk)'])
t=d.groupby('bucket',observed=True).y.agg(['mean','sum','size']).rename(columns={'mean':'default rate','sum':'defaults','size':'loans'})
t['share of all defaults']=t.defaults/t.defaults.sum(); display(t.round(3))
thr=np.quantile(p,0.8); print('Flagging top 20%%: recall=%.2f, precision=%.2f (base rate %.2f)'%(
    (d.y[d.p>=thr].sum()/d.y.sum()), d.y[d.p>=thr].mean(), d.y.mean()))

## 6. Conclusion

**Yes, moderately.** On the 2,000-loan held-out test set (10.5% default rate):

| model | ROC AUC | PR AUC |
|---|---|---|
| baseline (no skill) | 0.500 | 0.104 |
| **logistic regression** | **0.782** (95% CI ≈ 0.75–0.81) | **0.308** |
| gradient boosting | 0.772 | 0.299 |
| random forest | 0.760 | 0.306 |

- The simple logistic regression is as good as the tree models, so the signal looks mostly additive/linear; no benefit from added complexity here.
- Drivers: higher `debt_ratio`, lower `income_k`, shorter `employment_years` and larger `loan_amount_k` raise risk. `region`, `age`, `credit_lines` add essentially nothing.
- Practical use is **ranking**: the riskiest 20% of applicants contain ~53% of defaults (default rate ~27% vs 10% overall), and the lowest-risk 20% have under 1% default. Individual predictions are still weak (precision ~0.28 in the top quintile), so this suits triage or pricing, not automatic rejection.

**Caveats:** the numbers above come from a single random split of 8,000 rows (the CI is wide: roughly ±0.03 AUC); there is no date column, so I could not test stability over time; the data only contains funded loans' outcomes, so applicants who were declined are not represented (selection bias); and `region`/`age` should be reviewed for fair-lending implications before any use in decisions. Predicted probabilities are only roughly calibrated, so check the calibration plot before using them as absolute risk estimates.